# AAT6153 Chapter 2 — Hands-on 6
## Weather Forecasting with LSTM (Jena Climate)
**Slides:** Types of Sequence Data (68), Workflow (69), Hands-on 6 (79–84)

**Task:** predict the air temperature **24 hours ahead** from the past **5 days** (120 hourly readings × 14 sensor features). The data comes from the weather station of the Max Planck Institute for Biogeochemistry, Jena, the same kind of data an IoT weather station produces.

Workflow (slide 69): **prepare → split in time order → baseline → model → train → evaluate against the baseline**.

*Runtime: Runtime ▸ Change runtime type ▸ **T4 GPU** recommended.*

---
## Step 1: Download and load the data (slide 80)

In [ ]:
import numpy as np
import pandas as pd
import keras
from keras import layers

keras.utils.set_random_seed(7)
zip_path = keras.utils.get_file("jena_climate_2009_2016.csv.zip",
    "https://storage.googleapis.com/tensorflow/tf-keras-datasets/"
    "jena_climate_2009_2016.csv.zip")
df = pd.read_csv(zip_path)            # pandas reads the zipped CSV directly
df = df.iloc[5::6]                    # 10-min records -> 1 sample per hour
print(df.columns.tolist())
df.head()

In [ ]:
data = df.drop(columns=["Date Time"]).to_numpy("float32")   # 14 features
temp_col = 1                          # column index of "T (degC)"
print(data.shape)

In [ ]:
# Look at the temperature: a yearly cycle plus a daily cycle
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(data[:, temp_col], lw=0.4); ax1.set(title="Temperature 2009–2016 (hourly)", xlabel="Hour", ylabel="°C")
ax2.plot(data[:240, temp_col]); ax2.set(title="First 10 days", xlabel="Hour", ylabel="°C")
plt.show()

## Step 2: Normalise and create windowed datasets (slide 81)
Split **in time order** (50% train / 25% validation / 25% test) and never shuffle across the split, otherwise the model sees the future. Normalise with the **training** mean and standard deviation only.

In [ ]:
n = len(data)
n_train, n_val = int(0.5 * n), int(0.25 * n)
mean = data[:n_train].mean(axis=0)    # statistics from training data only
std  = data[:n_train].std(axis=0)
data = (data - mean) / std

seq_len, horizon = 120, 24            # past 5 days -> temperature in 24 h
delay = seq_len + horizon - 1

def make_ds(start, end, shuffle=False):
    return keras.utils.timeseries_dataset_from_array(
        data[:-delay], targets=data[delay:, temp_col],
        sequence_length=seq_len, batch_size=256, shuffle=shuffle,
        start_index=start, end_index=end)

train_ds = make_ds(0, n_train, shuffle=True)
val_ds   = make_ds(n_train, n_train + n_val)
test_ds  = make_ds(n_train + n_val, None)

for x, y in train_ds.take(1):
    print("Input batch:", x.shape, "  target batch:", y.shape)   # (256, 120, 14) (256,)

## Step 3: Compute a common-sense baseline (slide 82)
*"The temperature in 24 h equals the temperature now."* Because temperature has a strong daily cycle, this is hard to beat.

In [ ]:
def naive_mae(ds):                    # "tomorrow = same as now"
    err, count = 0.0, 0
    for x, y in ds:
        err += np.sum(np.abs(x[:, -1, temp_col].numpy() - y.numpy()))
        count += len(y)
    return err / count * std[temp_col]  # back to degrees Celsius

baseline_mae = naive_mae(test_ds)
print(f"Naive test MAE: {baseline_mae:.2f} °C")

## Step 4: Build, train and evaluate the LSTM (slide 83)
Many-to-one **regression**: the LSTM reads 120 hourly vectors of 14 features and outputs one number. MAE is converted back to °C by multiplying by the temperature standard deviation.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(seq_len, data.shape[-1])),   # (120, 14)
    layers.LSTM(32),
    layers.Dense(1)])                               # regression output
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                   restore_best_weights=True)
history = model.fit(train_ds, validation_data=val_ds, epochs=20,
                    callbacks=[es])
mae = model.evaluate(test_ds, verbose=0)[1] * std[temp_col]
print(f"LSTM test MAE: {mae:.2f} °C   (naive baseline: {baseline_mae:.2f} °C)")

## Step 5: Plot the learning curves (slide 84)

In [ ]:
plt.plot(history.history["mae"], "b", label="train")
plt.plot(history.history["val_mae"], "g", label="val")
plt.axhline(naive_mae(val_ds) / std[temp_col], color="r", ls="--", label="naive baseline (val)")
plt.xlabel("Epoch")
plt.ylabel("MAE (normalised)")
plt.title("LSTM temperature forecasting")
plt.legend(loc="best")
plt.show()

In [ ]:
# Forecast vs actual for the first 10 days of the test period
x_test, y_test = next(iter(test_ds))
pred = model.predict(x_test, verbose=0)[:240, 0] * std[temp_col] + mean[temp_col]
actual = y_test.numpy()[:240] * std[temp_col] + mean[temp_col]
naive = x_test.numpy()[:240, -1, temp_col] * std[temp_col] + mean[temp_col]

plt.figure(figsize=(14, 4))
plt.plot(actual, "k", label="actual (24 h later)")
plt.plot(pred, "g", label="LSTM forecast")
plt.plot(naive, "r:", label="naive forecast")
plt.xlabel("Hour"); plt.ylabel("°C"); plt.title("24-hour-ahead forecasts on the test set"); plt.legend(); plt.show()

---
## Exercises (slide 84)
Record the test MAE (°C) for each and compare with the naive baseline.

| # | Change | Test MAE (°C) |
|---|---|---|
| 1 | Does the LSTM beat the naive baseline? | |
| 2 | Replace `LSTM(32)` with `GRU(32)` | |
| 3 | Stack two recurrent layers (`return_sequences=True` on the first) | |
| 4 | Add `dropout=0.2, recurrent_dropout=0.2` | |
| 5 | Use only the temperature column (1 feature): `data[:, [temp_col]]`, then set `temp_col = 0` | |

In [ ]:
# Use this cell for your experiments